In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
import pyfirmata
import time

if __name__ == '__main__':
    board = pyfirmata.Arduino('COM9')
    print("Communication successfully started")
    
def controller(value):
    if value == 0:
        board.digital[13].write(0)
        time.sleep(1)
    else:
        board.digital[13].write(1)
        time.sleep(1)

model = load_model("D:/Final Year Project/Face_Mask_Detection_Project/Face Mask Detection Model.h5")

classes = ['Mask', 'No Mask']

GR_dict = {0: (0, 0, 255), 1: (0, 255, 0)}
rect_size = 4
cap = cv2.VideoCapture(0)

time_counter = 0  # Renamed the variable to avoid conflict with the 'time' module

haarcascade = cv2.CascadeClassifier('D:/Final Year Project/Face_Mask_Detection_Project/haarcascade_frontalface_default.xml')

while True:
    (rval, im) = cap.read()
    img = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    im = cv2.flip(img, 1, 1) 

    rerect_size = cv2.resize(im, (im.shape[1] // rect_size, im.shape[0] // rect_size))
    faces = haarcascade.detectMultiScale(rerect_size)
    for f in faces:
        (x, y, w, h) = [v * rect_size for v in f] 

        face_img = im[y:y+h, x:x+w]
        rerect_sized = cv2.resize(face_img, (150, 150))
        normalized = rerect_sized / 255.0
        reshaped = np.reshape(normalized, (1, 150, 150, 3))
        reshaped = np.vstack([reshaped])
        pred = model.predict(reshaped)

        if pred <= 0.50:
            time_counter += 1
            if time_counter >= 6:
                cv2.rectangle(im, (x, y), (x+w, y+h), GR_dict[1], 2)
                cv2.rectangle(im, (x, y-40), (x+w, y), GR_dict[1], -1)
                cv2.putText(im, classes[0], (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
                controller(0)
        else:
            time_counter += 1
            if time_counter >= 6:
                cv2.rectangle(im, (x, y), (x+w, y+h), GR_dict[0], 2)
                cv2.rectangle(im, (x, y-40), (x+w, y), GR_dict[0], -1)
                cv2.putText(im, classes[1], (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
                controller(1)
    controller(0)

    cv2.imshow('LIVE', im)
    key = cv2.waitKey(10)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Communication successfully started
1/1 [==============================] - 0s 34ms/step
